# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


In [6]:
class Matrix:
    def __init__(self, n, m=None):
        # Two init styles:
        # 1) Matrix(n, m) -> zeros
        # 2) Matrix(values) where values is list of lists

        def shape(self):
            return (self._n, self._m)

        if m is None:
            self._init_from_values(n)
        else:
            if not isinstance(n, int) or not isinstance(m, int):
                raise TypeError("Matrix(n,m): n and m must be ints")
            if n < 0 or m < 0:
                raise ValueError("Matrix(n,m): n and m must be >= 0")

            self._n = n  # rows
            self._m = m  # cols
            self._data = []

            i = 0
            while i < n:
                self._data.append([0.0] * m)
                i += 1

    def _init_from_values(self, values):
        if not isinstance(values, list):
            raise TypeError("Matrix(values): values must be list of lists")

        if len(values) == 0:
            self._n = 0
            self._m = 0
            self._data = []
            return

        if not isinstance(values[0], list):
            raise TypeError("Matrix(values): values must be list of lists")

        m = len(values[0])

        i = 0
        while i < len(values):
            if not isinstance(values[i], list):
                raise TypeError("Matrix(values): each row must be a list")
            if len(values[i]) != m:
                raise ValueError("Matrix(values): all rows must have same length")
            i += 1

        self._n = len(values)
        self._m = m
        self._data = []

        i = 0
        while i < self._n:
            row = []
            j = 0
            while j < self._m:
                row.append(float(values[i][j]))
                j += 1
            self._data.append(row)
            i += 1

2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

In [11]:
class Matrix:
    def __init__(self, values):
        if not isinstance(values, list) or (len(values) > 0 and not isinstance(values[0], list)):
            raise TypeError("Matrix(values): values must be list of lists")

        if len(values) == 0:
            self._n = 0
            self._m = 0
            self._data = []
            return

        m = len(values[0])
        i = 0
        while i < len(values):
            if len(values[i]) != m:
                raise ValueError("Matrix(values): all rows must have same length")
            i += 1

        self._n = len(values)
        self._m = m

        self._data = []
        i = 0
        while i < self._n:
            row = []
            j = 0
            while j < self._m:
                row.append(float(values[i][j]))
                j += 1
            self._data.append(row)
            i += 1

    def shape(self):
        return (self._n, self._m)

    def transpose(self):
        vals = []
        j = 0
        while j < self._m:
            row = []
            i = 0
            while i < self._n:
                row.append(self._data[i][j])
                i += 1
            vals.append(row)
            j += 1
        return Matrix(vals)

    def row(self, n):
        if not isinstance(n, int):
            raise TypeError("row(n): n must be int")
        return Matrix([self._data[n][:]])

    def column(self, n):
        if not isinstance(n, int):
            raise TypeError("column(n): n must be int")
        vals = []
        i = 0
        while i < self._n:
            vals.append([self._data[i][n]])
            i += 1
        return Matrix(vals)

    def to_list(self):
        out = []
        i = 0
        while i < self._n:
            out.append(self._data[i][:])
            i += 1
        return out

    def block(self, n_0, n_1, m_0, m_1):
        # columns: n_0 .. n_1-1
        # rows:    m_0 .. m_1-1
        vals = []
        i = m_0
        while i < m_1:
            row = []
            j = n_0
            while j < n_1:
                row.append(self._data[i][j])
                j += 1
            vals.append(row)
            i += 1
        return Matrix(vals)

    def __getitem__(self, key):
        # M[i][j] works because int key returns a row list
        # M[i,j] and slicing return a Matrix

        if isinstance(key, tuple):
            if len(key) != 2:
                raise TypeError("Use M[i,j] with exactly 2 indices")

            rkey = key[0]
            ckey = key[1]

            # scalar
            if isinstance(rkey, int) and isinstance(ckey, int):
                return self._data[rkey][ckey]

            # row indices
            if isinstance(rkey, slice):
                rstart, rstop, rstep = rkey.indices(self._n)
                rlist = []
                i = rstart
                while (i < rstop and rstep > 0) or (i > rstop and rstep < 0):
                    rlist.append(i)
                    i += rstep
            elif isinstance(rkey, int):
                rlist = [rkey]
            else:
                raise TypeError("Row index must be int or slice")

            # col indices
            if isinstance(ckey, slice):
                cstart, cstop, cstep = ckey.indices(self._m)
                clist = []
                j = cstart
                while (j < cstop and cstep > 0) or (j > cstop and cstep < 0):
                    clist.append(j)
                    j += cstep
            elif isinstance(ckey, int):
                clist = [ckey]
            else:
                raise TypeError("Column index must be int or slice")

            vals = []
            a = 0
            while a < len(rlist):
                i = rlist[a]
                row = []
                b = 0
                while b < len(clist):
                    j = clist[b]
                    row.append(self._data[i][j])
                    b += 1
                vals.append(row)
                a += 1

            return Matrix(vals)

        # single slice => rows slice, all columns
        if isinstance(key, slice):
            start, stop, step = key.indices(self._n)
            vals = []
            i = start
            while (i < stop and step > 0) or (i > stop and step < 0):
                vals.append(self._data[i][:])
                i += step
            return Matrix(vals)

        # single int => row list for M[i][j]
        if isinstance(key, int):
            return self._data[key]

        raise TypeError("Invalid index")

    def __repr__(self):
        return "Matrix(" + repr(self.to_list()) + ")"


# ---- tests ----
M = Matrix([[1,2,3],
            [4,5,6],
            [7,8,9]])

print(M.shape())          
print(M.transpose())     
print(M.row(1))          
print(M.column(2))        

(3, 3)
Matrix([[1.0, 4.0, 7.0], [2.0, 5.0, 8.0], [3.0, 6.0, 9.0]])
Matrix([[4.0, 5.0, 6.0]])
Matrix([[3.0], [6.0], [9.0]])


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [12]:
def constant(n, m, c):
    if not isinstance(n, int) or not isinstance(m, int):
        raise TypeError("constant(n,m,c): n and m must be int")
    if n < 0 or m < 0:
        raise ValueError("constant(n,m,c): n and m must be >= 0")

    c = float(c)

    vals = []
    i = 0
    while i < n:
        vals.append([c] * m)
        i += 1

    return Matrix(vals)

def zeros(n, m):
    return constant(n, m, 0.0)

def ones(n, m):
    return constant(n, m, 1.0)

def eye(n):
    if not isinstance(n, int):
        raise TypeError("eye(n): n must be int")
    if n < 0:
        raise ValueError("eye(n): n must be >= 0")

    vals = []
    i = 0
    while i < n:
        row = [0.0] * n
        row[i] = 1.0
        vals.append(row)
        i += 1

    return Matrix(vals)

print(constant(2,3,5))
print(zeros(2,3))
print(ones(2,3))
print(eye(4))


Matrix([[5.0, 5.0, 5.0], [5.0, 5.0, 5.0]])
Matrix([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0]])
Matrix([[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]])
Matrix([[1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]])


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [17]:
def _assert_matrix(self, N):
    if not isinstance(N, Matrix):
        raise TypeError("Argument must be a Matrix")

def _assert_same_shape(self, N):
    self._assert_matrix(N)
    if self._n != N._n or self._m != N._m:
        raise ValueError("Matrices must have the same shape")

def scalarmul(self, c):
    c = float(c)
    vals = []
    i = 0
    while i < self._n:
        row = []
        j = 0
        while j < self._m:
            row.append(self._data[i][j] * c)
            j += 1
        vals.append(row)
        i += 1
    return Matrix(vals)

def add(self, N):
    self._assert_same_shape(N)
    vals = []
    i = 0
    while i < self._n:
        row = []
        j = 0
        while j < self._m:
            row.append(self._data[i][j] + N._data[i][j])
            j += 1
        vals.append(row)
        i += 1
    return Matrix(vals)

def sub(self, N):
    self._assert_same_shape(N)
    vals = []
    i = 0
    while i < self._n:
        row = []
        j = 0
        while j < self._m:
            row.append(self._data[i][j] - N._data[i][j])
            j += 1
        vals.append(row)
        i += 1
    return Matrix(vals)

def element_mult(self, N):
    self._assert_same_shape(N)
    vals = []
    i = 0
    while i < self._n:
        row = []
        j = 0
        while j < self._m:
            row.append(self._data[i][j] * N._data[i][j])
            j += 1
        vals.append(row)
        i += 1
    return Matrix(vals)

def mat_mult(self, N):
    self._assert_matrix(N)
    if self._m != N._n:
        raise ValueError("mat_mult(): incompatible shapes")

    n = self._n
    m = self._m
    p = N._m

    vals = []
    i = 0
    while i < n:
        vals.append([0.0] * p)
        i += 1

    i = 0
    while i < n:
        j = 0
        while j < p:
            s = 0.0
            k = 0
            while k < m:
                s += self._data[i][k] * N._data[k][j]
                k += 1
            vals[i][j] = s
            j += 1
        i += 1

    return Matrix(vals)

def equals(self, N):
    if not isinstance(N, Matrix):
        return False
    if self._n != N._n or self._m != N._m:
        return False

    i = 0
    while i < self._n:
        j = 0
        while j < self._m:
            if self._data[i][j] != N._data[i][j]:
                return False
            j += 1
        i += 1
    return True

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


In [18]:
def __add__(self, other):
    return self.add(other)

def __sub__(self, other):
    return self.sub(other)

def __mul__(self, other):
    if isinstance(other, Matrix):
        return self.mat_mult(other)
    if isinstance(other, (int, float)) and not isinstance(other, bool):
        return self.scalarmul(other)
    raise TypeError("Unsupported * for Matrix")

def __rmul__(self, other):
    if isinstance(other, (int, float)) and not isinstance(other, bool):
        return self.scalarmul(other)
    raise TypeError("Unsupported * for Matrix")

def __eq__(self, other):
    return self.equals(other)

6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [31]:

A = Matrix([[1, 2],
            [0, 1]])

B = Matrix([[2, 0],
            [1, 3]])

C = Matrix([[0, 1],
            [4, 2]])

I = eye(2)

# (AB)C = A(BC)
left1  = (A * B) * C
right1 = A * (B * C)
print("(AB)C == A(BC):", left1 == right1)
print(" (AB)C =", left1)
print(" A(BC) =", right1)
print()

# A(B + C) = AB + AC
left2  = A * (B + C)
right2 = (A * B) + (A * C)
print("A(B+C) == AB+AC:", left2 == right2)
print(" A(B+C) =", left2)
print(" AB+AC  =", right2)
print()

# AB ≠ BA
AB = A * B
BA = B * A
print("AB != BA:", AB != BA)
print(" AB =", AB)
print(" BA =", BA)
print()

# AI = A
AI = A * I
print("AI == A:", AI == A)
print(" AI =", AI)
print(" A  =", A)

(AB)C == A(BC): True
 (AB)C = Matrix([[24.0, 16.0], [12.0, 7.0]])
 A(BC) = Matrix([[24.0, 16.0], [12.0, 7.0]])

A(B+C) == AB+AC: True
 A(B+C) = Matrix([[12.0, 11.0], [5.0, 5.0]])
 AB+AC  = Matrix([[12.0, 11.0], [5.0, 5.0]])

AB != BA: True
 AB = Matrix([[4.0, 6.0], [1.0, 3.0]])
 BA = Matrix([[2.0, 4.0], [1.0, 5.0]])

AI == A: True
 AI = Matrix([[1.0, 2.0], [0.0, 1.0]])
 A  = Matrix([[1.0, 2.0], [0.0, 1.0]])
